# Milestone 9 — Post-SFT evaluation

## Goal

Measure the incremental value of Ministral-3B SFT v1 against every major DomainAdapt baseline using the untouched 24-question `frozen_eval` split.

Milestone 8 established that SFT v1 did not improve the 12-question working gate: both the tuned and untuned context-engineered models passed 4/12. That does not replace the final frozen measurement. The working set is small, and the PRD requires an unbiased post-training comparison.

This notebook therefore answers one narrow question:

> With the same context-engineered system, did changing only the model weights improve frozen execution accuracy enough to justify adaptation?

No model or prompt selection happens here. This is the final confirmatory measurement.


## What is—and is not—deterministic

The evaluation is **controlled and reproducible**, but not perfectly bit-for-bit deterministic.

| Component | Property |
|---|---|
| Frozen question IDs | Deterministic and immutable |
| Prompt, schema context, deployment, and token limit | Fixed by hashes and configuration |
| Reference compatibility transformations | Deterministic and reference-only |
| SQLite execution | Deterministic for this fixed database |
| Ordered/unordered result comparison | Deterministic |
| Aggregation and run comparison | Deterministic from saved artifacts |
| Model generation | One fixed attempt, but hosted inference is not guaranteed bit-identical |
| API latency | Naturally variable |

The correct experimental control is therefore not “the model must emit identical bytes every time.” It is: **one predeclared request per frozen question, immediately persisted, with no cherry-picking, repair, or correctness retry.** Only requests that produce no model response because of infrastructure failure may be repeated.


## 1. Sealed evaluation contract

- Split: 24-question `frozen_eval`
- Model: deployed SFT v1 Ministral-3B
- System: selected Milestone 5 compact context and guardrails
- Output handling: strict removal of one surrounding SQL fence
- Candidate SQL repair: disabled
- Incorrect-answer retry: disabled
- Attempts: one valid model response per question
- Persistence: atomic save after every request
- Visibility during collection: progress counts only; no question IDs or outcomes
- Comparison: execution accuracy, executability, failure categories, latency, and token usage

The notebook refuses to run when the evaluation manifest, context-engineering configuration, deployment variable, or split membership does not match the expected contract.


In [1]:
from collections import Counter
from datetime import datetime, timezone
from hashlib import sha256
from pathlib import Path
from time import perf_counter
import json
import os
import sqlite3

from dotenv import load_dotenv

from domain_adapt.context_engineering import (
    load_context_engineering_config,
    request_context_engineered_sql,
)
from domain_adapt.evaluation import compare_sql_results
from domain_adapt.evaluation_analysis import (
    compare_runs,
    failure_category,
    index_results,
    summarize_results,
)
from domain_adapt.finch import make_reference_compatible


def find_repository_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / ".git").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repository root")


def sha256_file(path: Path) -> str:
    digest = sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def save_json_atomic(path: Path, value: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    temporary_path.write_text(
        json.dumps(value, indent=2) + "\n",
        encoding="utf-8",
    )
    temporary_path.replace(path)


REPOSITORY_ROOT = find_repository_root(Path.cwd())
DATA_DIR = REPOSITORY_ROOT / "data"
SEALED_RUN_DIR = DATA_DIR / "runs" / "sealed"

DB_PATH = DATA_DIR / "financial.sqlite"
DATASET_PATH = DATA_DIR / "finch_dataset.json"
MANIFEST_PATH = REPOSITORY_ROOT / "configs" / "eval_manifest_v1.json"
CONFIG_PATH = REPOSITORY_ROOT / "configs" / "slm_context_engineered_v1.json"

EXPECTED_CONFIG_SHA256 = (
    "574a775a2c91826a4d0073f78e84c40c3c4f5b6fbb61f656bd63f307252cdf68"
)
EXPECTED_INSTRUCTIONS_SHA256 = (
    "175e0489882789131bba788ad087f79e24a2207460d934778a16e04d7f265509"
)
EXPECTED_CONTEXT_SHA256 = (
    "634503b9d904730790555dfeabb2a81a920293a9d747416a04e372e122d7b987"
)

load_dotenv(REPOSITORY_ROOT / ".env", override=True)
print("Repository root:", REPOSITORY_ROOT)


Repository root: C:\Users\shchitt\Downloads\Projects\domain-adapt


## 2. Reconstruct and validate the frozen split

The code below verifies the complete evaluation contract before creating a client or making a model request. It loads only records whose IDs were frozen in Milestone 1 and proves that they remain disjoint from `working_dev`.


In [2]:
for required_path in (
    DB_PATH,
    DATASET_PATH,
    MANIFEST_PATH,
    CONFIG_PATH,
):
    if not required_path.exists():
        raise FileNotFoundError(required_path)

assert sha256_file(CONFIG_PATH) == EXPECTED_CONFIG_SHA256

context_config = load_context_engineering_config(CONFIG_PATH)
assert context_config["instructions_sha256"] == EXPECTED_INSTRUCTIONS_SHA256
assert context_config["schema_context_sha256"] == EXPECTED_CONTEXT_SHA256
assert context_config["generation"]["attempts_per_question"] == 1
assert context_config["generation"]["candidate_sql_repair"] is False
assert context_config["generation"]["retry_incorrect_answers"] is False

records = json.loads(DATASET_PATH.read_text(encoding="utf-8"))
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))

working_ids = set(manifest["selection"]["working_dev"]["ids"])
frozen_ids = set(manifest["selection"]["frozen_eval"]["ids"])
assert working_ids.isdisjoint(frozen_ids)

frozen_examples = sorted(
    [
        row for row in records
        if row["db_name"] == "bird"
        and row["db_id"] == "financial"
        and row["question_id"] in frozen_ids
    ],
    key=lambda row: row["question_id"],
)

assert len(frozen_examples) == 24
assert {row["question_id"] for row in frozen_examples} == frozen_ids

conn = sqlite3.connect(
    f"file:{DB_PATH.as_posix()}?mode=ro",
    uri=True,
)

sft_deployment_name = os.environ["AZURE_SFT_MODEL_DEPLOYMENT_NAME"].strip()
assert sft_deployment_name

print("Frozen questions:", len(frozen_examples))
print("Working/frozen overlap:", len(working_ids & frozen_ids))
print("Configuration SHA-256:", EXPECTED_CONFIG_SHA256)
print("SFT deployment:", sft_deployment_name)
print("Contract validation passed; no model request made.")


Frozen questions: 24
Working/frozen overlap: 0
Configuration SHA-256: 574a775a2c91826a4d0073f78e84c40c3c4f5b6fbb61f656bd63f307252cdf68
SFT deployment: Ministral-3B.ft-bd96ca624ae04994bab47135e9d12ebb-domain-adapt-
Contract validation passed; no model request made.


## 3. Bind the existing Foundry client

The same project client used in Milestone 8 is recreated from the configured Foundry project endpoint. A connectivity probe is intentionally omitted: the deployment already passed connectivity and Question 16 smoke tests. An extra probe would add no evaluation evidence.


In [3]:
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential


project_endpoint = os.environ["AZURE_AI_PROJECT_ENDPOINT"].strip()
assert project_endpoint

project_client = AIProjectClient(
    endpoint=project_endpoint,
    credential=DefaultAzureCredential(),
)
openai_client = project_client.get_openai_client()

print("Client created.")
print("No model request made.")


Client created.
No model request made.


## 4. Initialize or resume the sealed run

`RUN_FROZEN_EVALUATION` is the only switch that permits the 24 model requests. Review the contract above first, then change it to `True` for the official run.

The artifact is resumable. If the kernel stops after 10 valid results, rerunning these cells preserves those 10 and requests only the remaining 14. Duplicate question IDs are rejected.


In [5]:
INITIAL_RUN_PATH = (
    SEALED_RUN_DIR / "ministral_3b_sft_v1_frozen_eval_v2.json"
)
RUN_FROZEN_EVALUATION = True

run_contract = {
    "run_version": 1,
    "benchmark": manifest["benchmark"],
    "split": "frozen_eval",
    "variant": "ministral-3b-sft-v1-context-engineered-v1",
    "deployment_name": sft_deployment_name,
    "config_sha256": EXPECTED_CONFIG_SHA256,
    "instructions_sha256": EXPECTED_INSTRUCTIONS_SHA256,
    "schema_context_sha256": EXPECTED_CONTEXT_SHA256,
    "attempts_per_question": 1,
    "candidate_sql_repair": False,
    "retry_incorrect_answers": False,
    "output_normalization": context_config["output_normalization"],
}

if INITIAL_RUN_PATH.exists():
    sealed_run = json.loads(
        INITIAL_RUN_PATH.read_text(encoding="utf-8")
    )
    for key, expected_value in run_contract.items():
        assert sealed_run[key] == expected_value
    index_results(sealed_run["results"])
    print("Resuming saved results:", len(sealed_run["results"]))
elif RUN_FROZEN_EVALUATION:
    sealed_run = {
        **run_contract,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "results": [],
    }
    save_json_atomic(INITIAL_RUN_PATH, sealed_run)
    print("Initialized the sealed SFT run.")
else:
    sealed_run = None
    print("Frozen evaluation is disabled and no local artifact exists.")


Initialized the sealed SFT run.


## 5. Collect one valid response per frozen question

During collection, the notebook prints only progress counts. This prevents mid-run outcomes from influencing behavior. Each response is normalized, executed, compared, and atomically saved before the next request begins.

The broad exception handler is deliberate: an API or local failure is recorded rather than silently retried. Infrastructure-only failures are handled separately in the following section.


In [6]:
def collect_frozen_results(
    *,
    run: dict,
    examples: list[dict],
    output_path: Path,
) -> None:
    completed_ids = set(index_results(run["results"]))
    pending_examples = [
        example
        for example in examples
        if example["question_id"] not in completed_ids
    ]

    for example in pending_examples:
        question_id = example["question_id"]
        assert question_id in frozen_ids
        assert question_id not in working_ids
        started = perf_counter()

        try:
            response, raw_sql, normalized_sql, was_unwrapped = (
                request_context_engineered_sql(
                    client=openai_client,
                    deployment_name=sft_deployment_name,
                    question=example["question"],
                    config=context_config,
                )
            )
            reference_sql, rules = make_reference_compatible(example["SQL"])
            comparison = compare_sql_results(
                conn=conn,
                reference_sql=reference_sql,
                candidate_sql=normalized_sql,
            )
            result = {
                "question_id": question_id,
                "difficulty": example["difficulty"],
                "raw_generated_sql": raw_sql,
                "generated_sql": normalized_sql,
                "output_envelope_normalized": was_unwrapped,
                "passed": comparison.equivalent,
                "reason": comparison.reason,
                "expected_row_count": comparison.expected_row_count,
                "candidate_row_count": comparison.candidate_row_count,
                "reference_compatibility_rules": rules,
                "response_id": response.id,
                "response_model": response.model,
                "finish_reason": response.choices[0].finish_reason,
                "usage": response.usage.model_dump() if response.usage else {},
                "api_error": None,
            }
        except Exception as error:
            result = {
                "question_id": question_id,
                "difficulty": example["difficulty"],
                "raw_generated_sql": None,
                "generated_sql": None,
                "output_envelope_normalized": False,
                "passed": False,
                "reason": "model request failed",
                "expected_row_count": None,
                "candidate_row_count": None,
                "reference_compatibility_rules": None,
                "response_id": None,
                "response_model": None,
                "finish_reason": None,
                "usage": {},
                "api_error": f"{type(error).__name__}: {error}",
            }

        result["latency_seconds"] = round(perf_counter() - started, 3)
        run["results"].append(result)
        index_results(run["results"])
        save_json_atomic(output_path, run)

        print(f"Completed {len(run['results'])}/{len(frozen_examples)}")


if RUN_FROZEN_EVALUATION:
    assert sealed_run is not None
    collect_frozen_results(
        run=sealed_run,
        examples=frozen_examples,
        output_path=INITIAL_RUN_PATH,
    )
else:
    print("Frozen model requests skipped.")


Completed 1/24
Completed 2/24
Completed 3/24
Completed 4/24
Completed 5/24
Completed 6/24
Completed 7/24
Completed 8/24
Completed 9/24
Completed 10/24
Completed 11/24
Completed 12/24
Completed 13/24
Completed 14/24
Completed 15/24
Completed 16/24
Completed 17/24
Completed 18/24
Completed 19/24
Completed 20/24
Completed 21/24
Completed 22/24
Completed 23/24
Completed 24/24


## 6. Recover only infrastructure-missing responses

A correctness retry would bias the benchmark and is forbidden. An infrastructure retry is different: it is allowed only when the saved record proves that no model response was received:

- `api_error` is present;
- `response_id` is absent;
- generated SQL is absent; and
- usage is empty.

If such failures occurred, the completion artifact preserves every valid v1 response and replaces only the missing requests. Set `COMPLETE_INFRASTRUCTURE_FAILURES = True` only in that situation.


In [8]:
COMPLETED_RUN_PATH = (
    SEALED_RUN_DIR / "ministral_3b_sft_v1_frozen_eval_v2.json"
)
COMPLETE_INFRASTRUCTURE_FAILURES = False


def is_infrastructure_only_failure(result: dict) -> bool:
    return (
        result.get("api_error") is not None
        and result.get("response_id") is None
        and result.get("generated_sql") is None
        and not result.get("usage")
    )


if INITIAL_RUN_PATH.exists():
    initial_run = json.loads(INITIAL_RUN_PATH.read_text(encoding="utf-8"))
    initial_results = initial_run["results"]
    infrastructure_failures = [
        result
        for result in initial_results
        if is_infrastructure_only_failure(result)
    ]
    valid_results = [
        result
        for result in initial_results
        if not is_infrastructure_only_failure(result)
    ]
    print("Valid model responses:", len(valid_results))
    print("Infrastructure-missing responses:", len(infrastructure_failures))
else:
    initial_run = None
    infrastructure_failures = []
    valid_results = []
    print("Initial run artifact is not available.")


Valid model responses: 24
Infrastructure-missing responses: 0


In [9]:
if COMPLETE_INFRASTRUCTURE_FAILURES:
    assert initial_run is not None
    assert infrastructure_failures, (
        "No infrastructure-only failures require completion"
    )

    if COMPLETED_RUN_PATH.exists():
        completed_run = json.loads(
            COMPLETED_RUN_PATH.read_text(encoding="utf-8")
        )
        assert completed_run["source_run_sha256"] == sha256_file(
            INITIAL_RUN_PATH
        )
        index_results(completed_run["results"])
    else:
        completed_run = {
            **{
                key: value
                for key, value in initial_run.items()
                if key not in {"results", "summary"}
            },
            "run_version": 2,
            "source_run": INITIAL_RUN_PATH.name,
            "source_run_sha256": sha256_file(INITIAL_RUN_PATH),
            "completion_policy": (
                "Repeat only requests with no model response because of "
                "an authentication or infrastructure failure."
            ),
            "results": valid_results,
        }
        save_json_atomic(COMPLETED_RUN_PATH, completed_run)

    failed_ids = {
        result["question_id"] for result in infrastructure_failures
    }
    completed_ids = set(index_results(completed_run["results"]))
    completion_examples = [
        example
        for example in frozen_examples
        if example["question_id"] in failed_ids
        and example["question_id"] not in completed_ids
    ]

    collect_frozen_results(
        run=completed_run,
        examples=completion_examples,
        output_path=COMPLETED_RUN_PATH,
    )
else:
    print("Infrastructure completion skipped.")


Infrastructure completion skipped.


## 7. Select and validate the official SFT artifact

The v2 artifact is official only when infrastructure completion was required. Otherwise, v1 remains official. In either case, the official run must contain exactly 24 unique frozen IDs and zero API errors before aggregation begins.


In [10]:
candidate_paths = [COMPLETED_RUN_PATH, INITIAL_RUN_PATH]
official_sft_path = None
official_sft_run = None

for candidate_path in candidate_paths:
    if not candidate_path.exists():
        continue
    candidate_run = json.loads(candidate_path.read_text(encoding="utf-8"))
    candidate_results = candidate_run["results"]
    candidate_ids = set(index_results(candidate_results))
    if (
        len(candidate_results) == 24
        and candidate_ids == frozen_ids
        and all(result.get("api_error") is None for result in candidate_results)
    ):
        official_sft_path = candidate_path
        official_sft_run = candidate_run
        break

if official_sft_run is None:
    print("Official SFT evaluation is not complete yet.")
else:
    print("Official artifact:", official_sft_path)
    print("Responses:", len(official_sft_run["results"]))
    print("API errors: 0")


Official artifact: C:\Users\shchitt\Downloads\Projects\domain-adapt\data\runs\sealed\ministral_3b_sft_v1_frozen_eval_v2.json
Responses: 24
API errors: 0


## 8. Aggregate the official result

Everything from this point forward is deterministic and makes no model request. The shared analysis module computes the same metrics used for the working evaluation and rejects duplicate IDs.


In [11]:
if official_sft_run is not None:
    sft_summary = summarize_results(official_sft_run["results"])
    official_sft_run["summary"] = sft_summary
    save_json_atomic(official_sft_path, official_sft_run)
    official_sft_sha256 = sha256_file(official_sft_path)

    print("Official frozen SFT evaluation complete.")
    print(
        f"Overall: {sft_summary['passed']}/{sft_summary['total']} "
        f"({sft_summary['accuracy']:.1%})"
    )
    print(
        f"Executable: {sft_summary['executable']}/{sft_summary['total']} "
        f"({sft_summary['executable_rate']:.1%})"
    )
    print("By difficulty:", sft_summary["by_difficulty"])
    print("Failure categories:", sft_summary["failure_categories"])
    print("API errors:", sft_summary["api_errors"])
    print("Token usage:", sft_summary["token_usage"])
    print("Median latency:", sft_summary["median_latency_seconds"])
    print("Sealed-run SHA-256:", official_sft_sha256)
else:
    sft_summary = None
    official_sft_sha256 = None
    print("Aggregation skipped until the sealed run is complete.")


Official frozen SFT evaluation complete.
Overall: 7/24 (29.2%)
Executable: 16/24 (66.7%)
By difficulty: {'easy': {'passed': 5, 'total': 12, 'accuracy': 0.4166666666666667}, 'medium': {'passed': 2, 'total': 9, 'accuracy': 0.2222222222222222}, 'hard': {'passed': 0, 'total': 3, 'accuracy': 0.0}}
Failure categories: {'candidate_execution_failed': 8, 'unordered_results_differ': 6, 'column_count_differs': 2, 'ordered_results_differ': 1}
API errors: 0
Token usage: {'prompt_tokens': 32739, 'completion_tokens': 1567, 'total_tokens': 34306}
Median latency: 1.237
Sealed-run SHA-256: a65b5fdf2c7dbd8f8e7ae949147988e23fbc56a9ab726787030e6da2004abb3d


## 9. Compare all major variants

The project comparison separates three effects:

1. **Frontier capability:** GPT-5.2 with the original system.
2. **System contribution:** context engineering around untuned Ministral-3B.
3. **Weight adaptation:** SFT v1 with that exact same selected system.

The SFT-versus-context comparison is the cleanest causal comparison because only the deployed model weights changed.


In [12]:
VARIANT_PATHS = {
    "Frontier GPT-5.2": (
        SEALED_RUN_DIR / "gpt_5_2_minimal_frozen_eval_v1.json"
    ),
    "Bare Ministral-3B": (
        SEALED_RUN_DIR / "ministral_3b_minimal_frozen_eval_v1.json"
    ),
    "Context-engineered Ministral-3B": (
        SEALED_RUN_DIR
        / "ministral_3b_context_engineered_frozen_eval_v2.json"
    ),
}

variant_runs = {
    name: json.loads(path.read_text(encoding="utf-8"))
    for name, path in VARIANT_PATHS.items()
    if path.exists()
}

if official_sft_run is not None:
    variant_runs["SFT v1 + selected system"] = official_sft_run

variant_summaries = {
    name: summarize_results(run["results"])
    for name, run in variant_runs.items()
}

if variant_summaries:
    print(
        f"{'Variant':<38} {'Accuracy':>10} "
        f"{'Executable':>12} {'Tokens':>10} {'Latency':>9}"
    )
    print("-" * 85)
    for name, summary in variant_summaries.items():
        latency = summary["median_latency_seconds"]
        latency_text = "n/a" if latency is None else f"{latency:.3f}s"
        print(
            f"{name:<38} "
            f"{summary['passed']}/{summary['total']:<7} "
            f"{summary['executable']}/{summary['total']:<9} "
            f"{summary['token_usage']['total_tokens']:>10,} "
            f"{latency_text:>9}"
        )
else:
    print("No local sealed-run artifacts are available.")


Variant                                  Accuracy   Executable     Tokens   Latency
-------------------------------------------------------------------------------------
Frontier GPT-5.2                       11/24      24/24             9,081    1.827s
Bare Ministral-3B                      1/24      10/24             8,254    0.940s
Context-engineered Ministral-3B        7/24      17/24            34,241    1.079s
SFT v1 + selected system               7/24      16/24            34,306    1.237s


## 10. Attribute gains and regressions to SFT

Aggregate accuracy alone can hide churn. The transition analysis asks whether SFT created new passes, destroyed previous passes, or merely changed the SQL while leaving the outcome unchanged. It also compares the failure distributions before and after adaptation.

Question-level outcomes remain inside the artifact; the notebook prints only aggregate transition counts so the final report stays concise.


In [13]:
context_name = "Context-engineered Ministral-3B"
sft_name = "SFT v1 + selected system"

if context_name in variant_runs and sft_name in variant_runs:
    transition_rows = compare_runs(
        variant_runs[context_name]["results"],
        variant_runs[sft_name]["results"],
    )
    transition_counts = Counter(
        row["transition"] for row in transition_rows
    )
    print("Outcome transitions:", dict(transition_counts))

    baseline_by_id = index_results(
        variant_runs[context_name]["results"]
    )
    sft_by_id = index_results(variant_runs[sft_name]["results"])

    residual_transition_counts = Counter()
    for question_id in sorted(frozen_ids):
        baseline = baseline_by_id[question_id]
        tuned = sft_by_id[question_id]
        if baseline["passed"] or tuned["passed"]:
            continue
        residual_transition_counts[
            (
                failure_category(baseline),
                failure_category(tuned),
            )
        ] += 1

    print("Residual failure transitions:")
    for transition, count in residual_transition_counts.most_common():
        print(f"- {transition[0]} -> {transition[1]}: {count}")
else:
    transition_rows = []
    transition_counts = Counter()
    print("SFT/context transition analysis is not ready.")


Outcome transitions: {'still_passes': 7, 'still_fails': 17}
Residual failure transitions:
- candidate_execution_failed -> candidate_execution_failed: 6
- unordered_results_differ -> unordered_results_differ: 6
- column_count_differs -> candidate_execution_failed: 1
- column_count_differs -> column_count_differs: 1
- unordered_results_differ -> column_count_differs: 1
- candidate_execution_failed -> ordered_results_differ: 1
- unordered_results_differ -> candidate_execution_failed: 1


## 11. Make the adaptation decision

The decision is intentionally simple:

- **Supported:** SFT produces a meaningful accuracy gain over the context-engineered SLM without unacceptable regressions, cost, or latency.
- **Not supported:** accuracy ties or declines, or a tiny gain is outweighed by regressions and operating cost.

This is not a statistical-significance claim—the frozen set has only 24 examples. It is a practical V1 architecture decision supported by exact execution outcomes and transparent limitations.


In [14]:
DECISION_PATH = (
    DATA_DIR / "runs" / "fine_tuning" / "post_sft_evaluation_v1.json"
)

if context_name in variant_summaries and sft_name in variant_summaries:
    baseline_summary = variant_summaries[context_name]
    candidate_summary = variant_summaries[sft_name]

    accuracy_delta = (
        candidate_summary["accuracy"] - baseline_summary["accuracy"]
    )
    executable_delta = (
        candidate_summary["executable_rate"]
        - baseline_summary["executable_rate"]
    )
    token_delta = (
        candidate_summary["token_usage"]["total_tokens"]
        - baseline_summary["token_usage"]["total_tokens"]
    )

    adaptation_supported = accuracy_delta > 0
    decision = (
        "SFT v1 showed incremental frozen accuracy. Review gains, "
        "regressions, latency, and cost before promotion."
        if adaptation_supported
        else
        "SFT v1 did not improve frozen accuracy over the selected "
        "context-engineered baseline and is not a promotion candidate."
    )

    decision_record = {
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "baseline": context_name,
        "candidate": sft_name,
        "baseline_summary": baseline_summary,
        "candidate_summary": candidate_summary,
        "accuracy_delta": accuracy_delta,
        "executable_rate_delta": executable_delta,
        "total_token_delta": token_delta,
        "transition_counts": dict(transition_counts),
        "adaptation_supported": adaptation_supported,
        "decision": decision,
        "official_sft_artifact": official_sft_path.name,
        "official_sft_sha256": official_sft_sha256,
    }
    save_json_atomic(DECISION_PATH, decision_record)

    print("Accuracy delta:", f"{accuracy_delta:+.1%}")
    print("Executable-rate delta:", f"{executable_delta:+.1%}")
    print("Total-token delta:", f"{token_delta:+,}")
    print("Adaptation supported:", adaptation_supported)
    print("Decision:", decision)
    print("Decision artifact:", DECISION_PATH)
else:
    print("Decision deferred until both official runs are available.")


Accuracy delta: +0.0%
Executable-rate delta: -4.2%
Total-token delta: +65
Adaptation supported: False
Decision: SFT v1 did not improve frozen accuracy over the selected context-engineered baseline and is not a promotion candidate.
Decision artifact: C:\Users\shchitt\Downloads\Projects\domain-adapt\data\runs\fine_tuning\post_sft_evaluation_v1.json


## 12. How to interpret and report the result

After the official run, add the printed SFT metrics and decision to this notebook's final narrative before committing it.

Whatever the outcome, do not use the frozen failures to tune SFT v2 while continuing to describe this benchmark as untouched. The benchmark has now served its confirmatory purpose.

The next step is Milestone 10:

1. document what context engineering contributed;
2. document what SFT contributed—or failed to contribute;
3. explain the accuracy, latency, token, cost, and operational trade-offs;
4. extract the reusable domain-adaptation decision framework; and
5. update the README and architecture diagram.

A negative SFT result is not a failed project. DomainAdapt's core promise is to prove **when adaptation is useful and when the surrounding system or a stronger model is the better investment**.
